### Edge Torch Tutorial

In [3]:
import os
import ai_edge_torch
import copy
import numpy
import torch
import torchvision

from model_compression.src.utils.preprocessing import load_data
from model_compression.src.utils.model_setup import setup_model
from model_compression.src.Quantization.utils.model_setup import setup_qat_student_model
from torch.ao.quantization.quantize_pt2e import prepare_qat_pt2e, convert_pt2e, prepare_pt2e

2025-04-04 19:50:39.328194: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-04 19:50:39.336140: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1743810639.344961   26575 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1743810639.347627   26575 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1743810639.355326   26575 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [4]:
from torch.ao.quantization.quantizer.xnnpack_quantizer import (
  XNNPACKQuantizer,
  get_symmetric_quantization_config,
)
quantizer = XNNPACKQuantizer()
quantizer.set_global(get_symmetric_quantization_config())

In [7]:
model = torchvision.models.mobilenet_v2(weights="DEFAULT").eval()
sample_inputs = (torch.randn(1, 3, 224, 224),)
# torch_output = model(*sample_inputs)

In [8]:
exported_model = torch.export.export_for_training(model, sample_inputs).module()

In [9]:
prepared_model = prepare_pt2e(exported_model, quantizer)
print(prepared_model.graph)

graph():
    %features_0_0_weight : [num_users=1] = get_attr[target=features.0.0.weight]
    %activation_post_process_1 : [num_users=1] = call_module[target=activation_post_process_1](args = (%features_0_0_weight,), kwargs = {})
    %features_1_conv_0_0_weight : [num_users=1] = get_attr[target=features.1.conv.0.0.weight]
    %activation_post_process_4 : [num_users=1] = call_module[target=activation_post_process_4](args = (%features_1_conv_0_0_weight,), kwargs = {})
    %features_1_conv_1_weight : [num_users=1] = get_attr[target=features.1.conv.1.weight]
    %activation_post_process_7 : [num_users=1] = call_module[target=activation_post_process_7](args = (%features_1_conv_1_weight,), kwargs = {})
    %features_2_conv_0_0_weight : [num_users=1] = get_attr[target=features.2.conv.0.0.weight]
    %activation_post_process_9 : [num_users=1] = call_module[target=activation_post_process_9](args = (%features_2_conv_0_0_weight,), kwargs = {})
    %features_2_conv_1_0_weight : [num_users=1] = get_

/home/jacob-delgado/anaconda3/envs/ECG/lib/python3.12/site-packages/torch/fx/graph.py:1199: UserWarning: erase_node(batch_norm) on an already erased node
  warnings.warn(f"erase_node({to_erase}) on an already erased node")
/home/jacob-delgado/anaconda3/envs/ECG/lib/python3.12/site-packages/torch/fx/graph.py:1199: UserWarning: erase_node(batch_norm_1) on an already erased node
  warnings.warn(f"erase_node({to_erase}) on an already erased node")
/home/jacob-delgado/anaconda3/envs/ECG/lib/python3.12/site-packages/torch/fx/graph.py:1199: UserWarning: erase_node(batch_norm_2) on an already erased node
  warnings.warn(f"erase_node({to_erase}) on an already erased node")
/home/jacob-delgado/anaconda3/envs/ECG/lib/python3.12/site-packages/torch/fx/graph.py:1199: UserWarning: erase_node(batch_norm_3) on an already erased node
  warnings.warn(f"erase_node({to_erase}) on an already erased node")
/home/jacob-delgado/anaconda3/envs/ECG/lib/python3.12/site-packages/torch/fx/graph.py:1199: UserWarnin

In [10]:
quantized_model = convert_pt2e(prepared_model)
print(quantized_model)

GraphModule(
  (features): Module(
    (0): Module(
      (0): Module()
    )
    (1): Module(
      (conv): Module(
        (0): Module(
          (0): Module()
        )
        (1): Module()
      )
    )
    (2): Module(
      (conv): Module(
        (0): Module(
          (0): Module()
        )
        (1): Module(
          (0): Module()
        )
        (2): Module()
      )
    )
    (3): Module(
      (conv): Module(
        (0): Module(
          (0): Module()
        )
        (1): Module(
          (0): Module()
        )
        (2): Module()
      )
    )
    (4): Module(
      (conv): Module(
        (0): Module(
          (0): Module()
        )
        (1): Module(
          (0): Module()
        )
        (2): Module()
      )
    )
    (5): Module(
      (conv): Module(
        (0): Module(
          (0): Module()
        )
        (1): Module(
          (0): Module()
        )
        (2): Module()
      )
    )
    (6): Module(
      (conv): Module(
        (0): 

/home/jacob-delgado/anaconda3/envs/ECG/lib/python3.12/site-packages/torch/ao/quantization/utils.py:408: UserWarning: must run observer before calling calculate_qparams. Returning default values.
  warnings.warn(
/home/jacob-delgado/anaconda3/envs/ECG/lib/python3.12/site-packages/torch/ao/quantization/observer.py:1318: UserWarning: must run observer before calling calculate_qparams.                                    Returning default scale and zero point 
  warnings.warn(


In [ ]:
edge_model = ai_edge_torch.convert(model.eval(), sample_inputs)


In [ ]:
edge_output = edge_model(*sample_inputs)


In [ ]:
if (numpy.allclose(
    torch_output.detach().numpy(),
    edge_output,
    atol=1e-5,
    rtol=1e-5,
)):
    print("Inference result with Pytorch and TfLite was within tolerance")
else:
    print("Something wrong with Pytorch --> TfLite")


In [ ]:
edge_model.export('resnet.tflite')

In [ ]:
print(os.path.getsize("resnet.tflite") / 1e6)

In [ ]:
import os
import sys
import time
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

import torchvision
from torchvision import datasets
from torchvision.models.resnet import resnet18
import torchvision.transforms as transforms

# Set up warnings
import warnings
warnings.filterwarnings(
    action='ignore',
    category=DeprecationWarning,
    module=r'.*'
)
warnings.filterwarnings(
    action='default',
    module=r'torch.ao.quantization'
)

# Specify random seed for repeatable results
_ = torch.manual_seed(191009)

class AverageMeter(object):
    """Computes and stores the average and current value"""
    def __init__(self, name, fmt=':f'):
        self.name = name
        self.fmt = fmt
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

    def __str__(self):
        fmtstr = '{name} {val' + self.fmt + '} ({avg' + self.fmt + '})'
        return fmtstr.format(**self.__dict__)

def accuracy(output, target, topk=(1,)):
    """
    Computes the accuracy over the k top predictions for the specified
    values of k.
    """
    with torch.no_grad():
        maxk = max(topk)
        batch_size = target.size(0)

        _, pred = output.topk(maxk, 1, True, True)
        pred = pred.t()
        correct = pred.eq(target.view(1, -1).expand_as(pred))

        res = []
        for k in topk:
            correct_k = correct[:k].reshape(-1).float().sum(0, keepdim=True)
            res.append(correct_k.mul_(100.0 / batch_size))
        return res

def evaluate(model, criterion, data_loader, device):
    torch.ao.quantization.move_exported_model_to_eval(model)
    top1 = AverageMeter('Acc@1', ':6.2f')
    top5 = AverageMeter('Acc@5', ':6.2f')
    cnt = 0
    with torch.no_grad():
        for image, target in data_loader:
            image = image.to(device)
            target = target.to(device)
            output = model(image)
            loss = criterion(output, target)
            cnt += 1
            acc1, acc5 = accuracy(output, target, topk=(1, 5))
            top1.update(acc1[0], image.size(0))
            top5.update(acc5[0], image.size(0))
    print('')

    return top1, top5

def load_model(model_file):
    model = resnet18(pretrained=False)
    state_dict = torch.load(model_file, weights_only=True)
    model.load_state_dict(state_dict)
    return model

def print_size_of_model(model):
    if isinstance(model, torch.jit.RecursiveScriptModule):
        torch.jit.save(model, "temp.p")
    else:
        torch.jit.save(torch.jit.script(model), "temp.p")
    print("Size (MB):", os.path.getsize("temp.p")/1e6)
    os.remove("temp.p")

def prepare_data_loaders(data_path):
    normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                     std=[0.229, 0.224, 0.225])
    dataset = torchvision.datasets.ImageNet(
        data_path, split="train", transform=transforms.Compose([
            transforms.RandomResizedCrop(224),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            normalize,
        ]))
    dataset_test = torchvision.datasets.ImageNet(
        data_path, split="val", transform=transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            normalize,
        ]))

    train_sampler = torch.utils.data.RandomSampler(dataset)
    test_sampler = torch.utils.data.SequentialSampler(dataset_test)

    data_loader = torch.utils.data.DataLoader(
        dataset, batch_size=train_batch_size,
        sampler=train_sampler)

    data_loader_test = torch.utils.data.DataLoader(
        dataset_test, batch_size=eval_batch_size,
        sampler=test_sampler)

    return data_loader, data_loader_test

def train_one_epoch(model, criterion, optimizer, data_loader, device, ntrain_batches):
    # Note: do not call model.train() here, since this doesn't work on an exported model.
    # Instead, call `torch.ao.quantization.move_exported_model_to_train(model)`, which will
    # be added in the near future
    top1 = AverageMeter('Acc@1', ':6.2f')
    top5 = AverageMeter('Acc@5', ':6.2f')
    avgloss = AverageMeter('Loss', '1.5f')

    torch.ao.quantization.move_exported_model_to_train(model)

    cnt = 0
    for image, target in data_loader:
        start_time = time.time()
        print('.', end = '')
        cnt += 1
        image, target = image.to(device), target.to(device)
        output = model(image)
        loss = criterion(output, target)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        acc1, acc5 = accuracy(output, target, topk=(1, 5))
        top1.update(acc1[0], image.size(0))
        top5.update(acc5[0], image.size(0))
        avgloss.update(loss, image.size(0))
        if cnt >= ntrain_batches:
            print('Loss', avgloss.avg)

            print('Training: * Acc@1 {top1.avg:.3f} Acc@5 {top5.avg:.3f}'
                  .format(top1=top1, top5=top5))
            return

    print('Full imagenet train set:  * Acc@1 {top1.global_avg:.3f} Acc@5 {top5.global_avg:.3f}'
          .format(top1=top1, top5=top5))
    return

dataset = "SkinCancer"
batch_size = 32
learning_rate = 0.001
epochs = 1
save_dir = f"models/{dataset}/Quantized"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

os.makedirs(save_dir, exist_ok=True)

dataloaders = load_data(dataset=dataset, batch_size=batch_size)

train_batch_size = 32
eval_batch_size = 32

example_inputs = (next(iter(dataloaders["train"]))[0])
criterion = nn.CrossEntropyLoss()
# float_model = setup_model("mobilenet_v2", None, len(dataloaders["train"].dataset.classes))
float_model = setup_qat_student_model("mobilenet_v2", len(dataloaders["train"].dataset.classes))

In [ ]:
exported_model = torch.export.export_for_training(float_model, (example_inputs,)).module()

In [ ]:
from torch.ao.quantization.quantizer.xnnpack_quantizer import (
    XNNPACKQuantizer,
    get_symmetric_quantization_config,
)
quantizer = XNNPACKQuantizer()
quantizer.set_global(get_symmetric_quantization_config(is_qat=True))

In [ ]:
prepared_model = prepare_qat_pt2e(exported_model, quantizer)
print(prepared_model)
prepared_model.to(device)

In [ ]:
num_epochs = 10
num_train_batches = 20
num_eval_batches = 20
num_observer_update_epochs = 4
num_batch_norm_update_epochs = 3
num_epochs_between_evals = 2
optimizer = torch.optim.Adam(prepared_model.parameters(), lr=learning_rate)

# QAT takes time and one needs to train over a few epochs.
# Train and check accuracy after each epoch
for epoch in range(num_epochs):
    train_one_epoch(prepared_model, criterion, optimizer, dataloaders['train'], "cuda", num_train_batches)

    # Optionally disable observer/batchnorm stats after certain number of epochs
    if epoch >= num_observer_update_epochs:
        print("Disabling observer for subseq epochs, epoch = ", epoch)
        prepared_model.apply(torch.ao.quantization.disable_observer)
    if epoch >= num_batch_norm_update_epochs:
        print("Freezing BN for subseq epochs, epoch = ", epoch)
        for n in prepared_model.graph.nodes:
            # Args: input, weight, bias, running_mean, running_var, training, momentum, eps
            # We set the `training` flag to False here to freeze BN stats
            if n.target in [
                torch.ops.aten._native_batch_norm_legit.default,
                torch.ops.aten.cudnn_batch_norm.default,
            ]:
                new_args = list(n.args)
                new_args[5] = False
                n.args = new_args
        prepared_model.recompile()

    # Check the quantized accuracy every N epochs
    # Note: If you wish to just evaluate the QAT model (not the quantized model),
    # then you can just call `torch.ao.quantization.move_exported_model_to_eval/train`.
    # However, the latter API is not ready yet and will be available in the near future.
    # if (epoch + 1) % num_epochs_between_evals == 0:
    #     torch.ao.quantization.move_exported_model_to_train(prepared_model)
    #     prepared_model_copy = copy.deepcopy(prepared_model)
    #     quantized_model = convert_pt2e(prepared_model_copy)
    #     top1, top5 = evaluate(quantized_model, criterion, dataloaders["test"], neval_batches=num_eval_batches)
    #     print('Epoch %d: Evaluation accuracy on %d images, %2.2f' % (epoch, num_eval_batches * eval_batch_size, top1.avg))

In [ ]:
prepared_model.print_readable()

In [ ]:
example_inputs = next(iter(dataloaders["train"]))[0].to("cpu")
populated_model = prepared_model.to("cpu")
populated_model(example_inputs)
quantized_model = convert_pt2e(populated_model, fold_quantize=False)
print(quantized_model)

In [ ]:
example_inputs = next(iter(dataloaders["train"]))[0].to("cpu")
torch.ao.quantization.move_exported_model_to_eval(quantized_model)
edge_model = ai_edge_torch.convert(quantized_model, (example_inputs,))
edge_model.export('test.tflite')

In [ ]:
print(os.path.getsize("test.tflite") / 1e6)

In [ ]:
import model_explorer
model_explorer.visualize('resnet.tflite')


In [ ]:
print(os.path.getsize("models/SkinCancer/Quantized/quantized_student_state.pth") / 1e6)
print(os.path.getsize("models/SkinCancer/Quantized/mobilenet_v2_quantized.pth") / 1e6)

In [ ]:
# TODO use quantization stuff from AI edge instead of normal PyTorch